In [1]:
%pip install cohere pinecone datasets python-dotenv

INFO: pip is looking at multiple versions of pydantic to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 19.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 29.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 642.6/642.6 kB 21.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 38.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 20.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 36.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 50.0 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 51.7 MB/s  0:00:006m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 43.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 45.3 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34/34 [cohere]33/34 [cohe

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

base_dir = Path('/workspaces/Notebook-LLMs/pincone')
env_file = base_dir / '.env'
example_file = base_dir / '.env.example'

loaded = False
if env_file.is_file():
    loaded = load_dotenv(env_file)
if not loaded and example_file.is_file():
    loaded = load_dotenv(example_file)

print('dotenv loaded:', loaded)
print('COHERE_API_KEY found:', bool(os.getenv('COHERE_API_KEY')))

dotenv loaded: True
PINECONE_API_KEY found: True


In [ ]:
import cohere

co = cohere.Client(api_key=os.getenv('COHERE_API_KEY'))
print('Cohere client initialized:', co is not None)

In [11]:
# The Hugging Face TREC dataset loader now relies on deprecated dataset scripts.
# Use a small local TREC-style sample so the notebook stays runnable without that dependency.
trec = {
    'text': [
        'What is the capital of France?',
        'Who wrote Pride and Prejudice?',
        'How tall is Mount Everest?',
        'What causes rainbows?',
        'When was the Eiffel Tower built?',
        'Which planet is known as the Red Planet?',
        'What is the largest ocean on Earth?',
        'Who invented the telephone?',
        'What is photosynthesis?',
        'How many continents are there on Earth?'
    ]
}

print(f"Loaded {len(trec['text'])} local TREC-style rows")

Loaded 10 local TREC-style rows


In [12]:
embeds = co.embed(
    texts=trec['text'],
    model='embed-english-v3.0',
    input_type='search_document',
    truncate='END'
).embeddings

In [13]:
import numpy as np

shape = np.array(embeds).shape
print(shape)

(10, 1024)


In [14]:
from pinecone import Pinecone, ServerlessSpec
pc = Pinecone()

index_name = 'cohere-pinecone-trec' 
if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=shape[1],
        metric="cosine",
        spec=ServerlessSpec(
            cloud='aws', 
            region='us-east-1'
        ) 
    )

# connect to index
index = pc.Index(index_name)

In [15]:
batch_size = 128

ids = [str(i) for i in range(shape[0])]
meta = [{'text': text} for text in trec['text']]
to_upsert = list(zip(ids, embeds, meta))

for i in range(0, shape[0], batch_size):
    i_end = min(i+batch_size, shape[0])
    index.upsert(vectors=to_upsert[i:i_end])

print(index.describe_index_stats())


{'_response_info': {'raw_headers': {'connection': 'keep-alive',
                                    'content-length': '184',
                                    'content-type': 'application/json',
                                    'date': 'Mon, 13 Apr 2026 18:27:30 GMT',
                                    'grpc-status': '0',
                                    'server': 'envoy',
                                    'x-envoy-upstream-service-time': '46',
                                    'x-pinecone-request-latency-ms': '46',
                                    'x-pinecone-response-duration-ms': '47'}},
 'dimension': 1024,
 'index_fullness': 0.0,
 'memoryFullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'__default__': {'vector_count': 10}},
 'storageFullness': 0.0,
 'total_vector_count': 10,
 'vector_type': 'dense'}


In [16]:
query = "What caused the 1929 Great Depression?"

xq = co.embed(
    texts=[query],
    model='embed-english-v3.0',
    input_type='search_query',
    truncate='END'
).embeddings

print(np.array(xq).shape)

res = index.query(vector=xq, top_k=5, include_metadata=True)

(1, 1024)


In [17]:
for match in res['matches']:
    print(f"{match['score']:.2f}: {match['metadata']['text']}")

0.27: What causes rainbows?
0.20: When was the Eiffel Tower built?
0.20: Who invented the telephone?
0.17: Who wrote Pride and Prejudice?
0.14: What is the capital of France?
